In [ ]:
%pip -q install -U "openai>=1.0.0" datasets rapidfuzz evaluate sacrebleu


In [1]:
import os

TMX_ROOT = r"J:\FINAL PROJECT"  # SAME as your Marian notebook
TMX_EN_ES = os.path.join(TMX_ROOT, "data", "en-es.tmx")
TMX_EN_PT = os.path.join(TMX_ROOT, "data", "en-pt.tmx")

PROJECT_DIR = "./hybrid_gpt_ft_scielo"
os.makedirs(PROJECT_DIR, exist_ok=True)

SEED = 42
MAX_TRAIN = 800
MAX_VALID = 200
SPLIT_RATIO = 0.90

PAIR = "en-pt"   # run once with "en-es", then set to "en-pt" and rerun from Cell 8 onward
BASE_MODEL = "gpt-4.1-mini-2025-04-14"


In [2]:
import re, json, time, random
from pathlib import Path
from typing import Dict, Tuple, Optional

import numpy as np
import xml.etree.ElementTree as ET

from datasets import Dataset
from openai import OpenAI

random.seed(SEED); np.random.seed(SEED)
client = OpenAI()  # requires OPENAI_API_KEY in your environment


j:\FINAL PROJECT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
_LANG_ALIASES = {
    "en": {"en", "eng", "en-us", "en-gb"},
    "es": {"es", "spa", "es-es", "es-la", "es-mx"},
    "pt": {"pt", "por", "pt-pt", "pt-br"},
}

def _norm_lang(code: str) -> str:
    c = (code or "").lower()
    for k, aliases in _LANG_ALIASES.items():
        if c in aliases:
            return k
    m = re.match(r"^([a-z]{2})[-_][a-z]{2}$", c)
    if m and m.group(1) in _LANG_ALIASES:
        return m.group(1)
    return c

def _extract_pair_from_tu(tu_elem, src_key, tgt_key):
    texts = {}
    for tuv in tu_elem.findall(".//tuv"):
        lang = _norm_lang(
            tuv.attrib.get("{http://www.w3.org/XML/1998/namespace}lang")
            or tuv.attrib.get("lang") or ""
        )
        seg = tuv.find(".//seg")
        if seg is not None and seg.text is not None:
            texts[lang] = seg.text.strip()
    if src_key in texts and tgt_key in texts:
        return texts[src_key], texts[tgt_key]
    return None

def read_tmx_pairs(path: str, src_key: str, tgt_key: str, max_samples: Optional[int] = None):
    if not os.path.exists(path):
        raise FileNotFoundError(f"TMX not found: {path}")
    src, tgt = [], []
    for event, elem in ET.iterparse(path, events=("end",)):
        if elem.tag.lower().endswith("tu"):
            pair = _extract_pair_from_tu(elem, src_key, tgt_key)
            if pair:
                s, t = pair
                if s and t:
                    src.append(s)
                    tgt.append(t)
                    if max_samples and len(src) >= max_samples:
                        break
            elem.clear()
    return src, tgt


In [5]:
TAG = re.compile(r"</?[^>]+>")
WS  = re.compile(r"\s+")

def _norm_text(x: str) -> str:
    x = x or ""
    x = TAG.sub(" ", x)
    x = WS.sub(" ", x)
    return x.strip()

def global_clean_and_dedup(src_all, tgt_all, min_ratio=0.5, max_ratio=1.8, min_len=5):
    rows = []
    for s, t in zip(src_all, tgt_all):
        s2, t2 = _norm_text(s), _norm_text(t)
        if not s2 or not t2:
            continue
        sw, tw = s2.split(), t2.split()
        if len(sw) < min_len or len(tw) < min_len:
            continue
        r = len(tw) / max(1, len(sw))
        if r < min_ratio or r > max_ratio:
            continue
        rows.append((s2, t2))

    seen = set()
    uniq = []
    for s2, t2 in rows:
        if s2 in seen:
            continue
        seen.add(s2)
        uniq.append((s2, t2))

    return [s for s,_ in uniq], [t for _,t in uniq]

def split_to_datasets(src_clean, tgt_clean, split_ratio=0.9, seed=42, max_train=None, max_valid=None):
    idxs = list(range(len(src_clean)))
    rng = random.Random(seed); rng.shuffle(idxs)
    cut = int(len(idxs) * split_ratio)
    train_idx, valid_idx = idxs[:cut], idxs[cut:]

    if max_train is not None:
        train_idx = train_idx[:max_train]
    if max_valid is not None:
        valid_idx = valid_idx[:max_valid]

    def build(idxs):
        return Dataset.from_dict({
            "src_text": [src_clean[i] for i in idxs],
            "tgt_text": [tgt_clean[i] for i in idxs],
        })
    return build(train_idx), build(valid_idx)


In [6]:
print("EN-ES TMX:", os.path.abspath(TMX_EN_ES))
src_es_all, tgt_es_all = read_tmx_pairs(TMX_EN_ES, "en", "es")
src_es_clean, tgt_es_clean = global_clean_and_dedup(src_es_all, tgt_es_all)
train_es, valid_es = split_to_datasets(src_es_clean, tgt_es_clean, split_ratio=SPLIT_RATIO, seed=SEED,
                                       max_train=MAX_TRAIN, max_valid=MAX_VALID)

print("EN-PT TMX:", os.path.abspath(TMX_EN_PT))
src_pt_all, tgt_pt_all = read_tmx_pairs(TMX_EN_PT, "en", "pt")
src_pt_clean, tgt_pt_clean = global_clean_and_dedup(src_pt_all, tgt_pt_all)
train_pt, valid_pt = split_to_datasets(src_pt_clean, tgt_pt_clean, split_ratio=SPLIT_RATIO, seed=SEED,
                                       max_train=MAX_TRAIN, max_valid=MAX_VALID)

print("EN→ES:", len(train_es), len(valid_es))
print("EN→PT:", len(train_pt), len(valid_pt))


EN-ES TMX: J:\FINAL PROJECT\data\en-es.tmx
EN-PT TMX: J:\FINAL PROJECT\data\en-pt.tmx
EN→ES: 800 200
EN→PT: 800 200


In [9]:
EQ_PATTERN   = re.compile(r"\b(?:log|ln|exp|sin|cos|tan)\s*\([^)]*\)")
LATEX_EQ     = re.compile(r"\$[^$]+\$")
CIT_PATTERN  = re.compile(r"[A-Z][a-z]+ et al\.\s*\(\d{4}\)")
PERC_PATTERN = re.compile(r"\d+(?:[.,]\d+)?\s*%")
NUM_PATTERN  = re.compile(r"\d+(?:[.,]\d+)?")
PROTECT_PATTERNS = [LATEX_EQ, EQ_PATTERN, CIT_PATTERN, PERC_PATTERN, NUM_PATTERN]

def rbmt_protect(text: str) -> Tuple[str, Dict[str, str]]:
    spans = []
    for pat in PROTECT_PATTERNS:
        for m in pat.finditer(text):
            spans.append((m.start(), m.end(), m.group(0)))
    spans.sort(key=lambda x: (x[0], -(x[1]-x[0])))

    non_over, last_end = [], -1
    for s,e,v in spans:
        if s >= last_end:
            non_over.append((s,e,v)); last_end = e

    out, cur, mp, k = [], 0, {}, 0
    for s,e,v in non_over:
        if cur < s:
            out.append(text[cur:s])
        ph = f"<P{k}>"; k += 1
        mp[ph] = v
        out.append(ph)
        cur = e
    if cur < len(text):
        out.append(text[cur:])
    return "".join(out), mp

def rbmt_restore(text: str, mp: Dict[str, str]) -> str:
    for ph, val in mp.items():
        text = text.replace(ph, val)
    return text

RBMT_LEXICON_ES = {
    "systematic review": "revisión sistemática",
    "confidence interval": "intervalo de confianza",
}
RBMT_LEXICON_PT = {
    "systematic review": "revisão sistemática",
    "confidence interval": "intervalo de confiança",
}

def rbmt_apply_lexicon(text: str, lang: str) -> str:
    lex = RBMT_LEXICON_ES if lang == "es" else RBMT_LEXICON_PT
    for src, tgt in lex.items():
        text = text.replace(src, tgt)
    return text


In [8]:
from rapidfuzz import process, fuzz

if PAIR == "en-es":
    train_ds, valid_ds = train_es, valid_es
    LANG, LANG_NAME = "es", "Spanish"
elif PAIR == "en-pt":
    train_ds, valid_ds = train_pt, valid_pt
    LANG, LANG_NAME = "pt", "Portuguese"
else:
    raise ValueError("PAIR must be 'en-es' or 'en-pt'")

tm = list(zip(train_ds["src_text"], train_ds["tgt_text"]))
src_tm = [s for s,_ in tm]

def tm_lookup(text: str, threshold: int = 98) -> Tuple[Optional[str], float]:
    best = process.extractOne(text, src_tm, scorer=fuzz.token_sort_ratio)
    if best is None:
        return None, 0.0
    best_src, score, idx = best
    if score >= threshold:
        return tm[idx][1], float(score)
    return None, float(score)

print("Using", PAIR, "| Train:", len(train_ds), "Valid:", len(valid_ds))


Using en-pt | Train: 800 Valid: 200


In [10]:
SYSTEM = (
    "You are a professional scientific translator. "
    "Output only the translation text. Preserve placeholders like <P0> exactly."
)

def make_record(src: str, tgt: str) -> dict:
    src_p, _ = rbmt_protect(src)
    tgt_p, _ = rbmt_protect(tgt)
    return {"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Translate English to {LANG_NAME}:\n{src_p}"},
        {"role": "assistant", "content": tgt_p},
    ]}

def write_jsonl(ds: Dataset, path: str):
    with open(path, "w", encoding="utf-8") as f:
        for ex in ds:
            f.write(json.dumps(make_record(ex["src_text"], ex["tgt_text"]), ensure_ascii=False) + "\n")

train_path = os.path.join(PROJECT_DIR, f"train_{PAIR}.jsonl")
valid_path = os.path.join(PROJECT_DIR, f"valid_{PAIR}.jsonl")
write_jsonl(train_ds, train_path)
write_jsonl(valid_ds, valid_path)

print("Wrote:", train_path)
print("Wrote:", valid_path)


Wrote: ./hybrid_gpt_ft_scielo\train_en-pt.jsonl
Wrote: ./hybrid_gpt_ft_scielo\valid_en-pt.jsonl


In [11]:
from pathlib import Path


In [12]:
train_file = client.files.create(file=Path(train_path), purpose="fine-tune")
valid_file = client.files.create(file=Path(valid_path), purpose="fine-tune")

job = client.fine_tuning.jobs.create(
    model=BASE_MODEL,
    training_file=train_file.id,
    validation_file=valid_file.id,
    suffix=f"hybrid-{PAIR}"
)
print("Job:", job.id, "Status:", job.status)


Job: ftjob-ZZAqLCLHKnPYR2AFomgbph7E Status: validating_files


In [13]:
def wait(job_id: str, sleep_s: int = 20):
    while True:
        j = client.fine_tuning.jobs.retrieve(job_id)
        print("status:", j.status)
        if j.status in ("succeeded", "failed", "cancelled"):
            return j
        time.sleep(sleep_s)

final_job = wait(job.id)
FT_MODEL = getattr(final_job, "fine_tuned_model", None)
print("Fine-tuned model:", FT_MODEL)


status: validating_files
status: validating_files
status: validating_files
status: validating_files
status: validating_files
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
status: queued
statu

In [14]:
def gpt_ft_translate_one(protected_src: str) -> str:
    resp = client.responses.create(
        model=FT_MODEL,
        input=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Translate English to {LANG_NAME}:\n{protected_src}"}
        ],
    )
    return (resp.output_text or "").strip()

def hybrid_translate(text: str, tm_threshold: int = 98) -> Tuple[str, dict]:
    protected, mp = rbmt_protect(text)
    protected = rbmt_apply_lexicon(protected, LANG)

    tm_out, score = tm_lookup(protected, threshold=tm_threshold)
    if tm_out is not None:
        return rbmt_restore(tm_out, mp), {"mode": "tm", "tm_score": score, "placeholders": len(mp)}

    out = gpt_ft_translate_one(protected)
    return rbmt_restore(out, mp), {"mode": "gpt_ft", "tm_score": score, "placeholders": len(mp)}


In [16]:
import evaluate
import numpy as np

bleu_metric = evaluate.load("sacrebleu")
ter_metric  = evaluate.load("ter")


In [17]:
def pure_ft_translate(text: str) -> str:
    # Keep only placeholder protection to avoid breaking equations/citations
    protected, mp = rbmt_protect(text)

    resp = client.responses.create(
        model=FT_MODEL,
        input=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Translate English to {LANG_NAME}:\n{protected}"}
        ],
    )
    out = (resp.output_text or "").strip()
    return rbmt_restore(out, mp)

def run_eval(valid_ds, pred_fn, max_n=None):
    n = len(valid_ds) if max_n is None else min(max_n, len(valid_ds))
    preds, refs = [], []

    for i in range(n):
        src = valid_ds[i]["src_text"]
        ref = valid_ds[i]["tgt_text"]
        hyp = pred_fn(src)

        preds.append(hyp)
        refs.append(ref)

    # sacrebleu/ter expect references as list[list[str]]
    refs_ll = [[r] for r in refs]

    bleu = bleu_metric.compute(
        predictions=preds,
        references=refs_ll,
        use_effective_order=True
    )["score"]

    ter = ter_metric.compute(
        predictions=preds,
        references=refs_ll
    )["score"]

    exact = float(np.mean([p == r for p, r in zip(preds, refs)]) * 100.0)

    return {
        "n": n,
        "bleu": float(bleu),
        "ter": float(ter),
        "exact_match_%": exact
    }


In [18]:
res_pure = run_eval(valid_ds, pred_fn=pure_ft_translate, max_n=20)
print(f"{PAIR} Pure FT (n=20) -> BLEU {res_pure['bleu']:.2f} | TER {res_pure['ter']:.2f}")

res_hybrid = run_eval(valid_ds, pred_fn=lambda s: hybrid_translate(s, tm_threshold=98)[0], max_n=20)
print(f"{PAIR} Hybrid  (n=20) -> BLEU {res_hybrid['bleu']:.2f} | TER {res_hybrid['ter']:.2f}")


en-pt Pure FT (n=20) -> BLEU 35.25 | TER 50.76
en-pt Hybrid  (n=20) -> BLEU 35.31 | TER 51.99


In [19]:
# Pure fine-tuned GPT model
res_pure = run_eval(valid_ds, pred_fn=pure_ft_translate, max_n=None)
print(f"{PAIR} Pure FT -> BLEU {res_pure['bleu']:.2f} | TER {res_pure['ter']:.2f} | Exact {res_pure['exact_match_%']:.2f}%")

# Hybrid pipeline (TM + lexicon + FT fallback)
res_hybrid = run_eval(valid_ds, pred_fn=lambda s: hybrid_translate(s, tm_threshold=98)[0], max_n=None)
print(f"{PAIR} Hybrid  -> BLEU {res_hybrid['bleu']:.2f} | TER {res_hybrid['ter']:.2f} | Exact {res_hybrid['exact_match_%']:.2f}%")

res_pure, res_hybrid


en-pt Pure FT -> BLEU 37.98 | TER 49.27 | Exact 1.00%
en-pt Hybrid  -> BLEU 37.86 | TER 49.31 | Exact 1.50%


({'n': 200,
  'bleu': 37.97801195382442,
  'ter': 49.27269687343254,
  'exact_match_%': 1.0},
 {'n': 200,
  'bleu': 37.85739262023232,
  'ter': 49.306136097642536,
  'exact_match_%': 1.5})

In [ ]:
# Smoke test for your fine-tuned EN→ES model
# - Verifies the model exists and is callable
# - Runs a few tiny translations
# - Fails fast with useful prints

import os
from openai import OpenAI

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
client = OpenAI()

FT_MODEL = "ft:gpt-4.1-mini-2025-04-14:personal:hybrid-en-es:DCX2w9hX"

tests = [
    ("We used a convolutional neural network for image classification.", "ES"),
    ("The dataset contains 800 training samples and 200 validation samples.", "ES"),
    ("Figure 2 shows the results of the ablation study.", "ES"),
    ("The patient was treated with antibiotics for 7 days.", "ES"),
]

def translate_smoke(src: str, tgt_lang: str = "ES") -> str:
    resp = client.chat.completions.create(
        model=FT_MODEL,
        messages=[
            {"role": "system", "content": f"You are a translation engine. Translate to {tgt_lang}. Output only the translation."},
            {"role": "user", "content": src},
        ],
        temperature=0,
        max_tokens=300,
    )
    return resp.choices[0].message.content.strip()

# 1) Single-call sanity check (fast fail)
print("Model:", FT_MODEL)
out = translate_smoke("This is a smoke test.", "ES")
print("OK ->", out)

# 2) Batch smoke tests
for i, (src, tgt) in enumerate(tests, 1):
    hyp = translate_smoke(src, tgt)
    print(f"\n[{i}] SRC: {src}\n    HYP: {hyp}")

print("\nSmoke test finished.")


Model: ft:gpt-4.1-mini-2025-04-14:personal:hybrid-en-es:DCX2w9hX
OK -> Esta es una prueba de humo.

[1] SRC: We used a convolutional neural network for image classification.
    HYP: Usamos una red neuronal convolucional para la clasificación de imágenes.

[2] SRC: The dataset contains 800 training samples and 200 validation samples.
    HYP: El conjunto de datos contiene 800 muestras de entrenamiento y 200 muestras de validación.

[3] SRC: Figure 2 shows the results of the ablation study.
    HYP: La Figura 2 muestra los resultados del estudio de ablación.

[4] SRC: The patient was treated with antibiotics for 7 days.
    HYP: La paciente recibió tratamiento antibiótico por 7 días.

Smoke test finished.
